In [1]:
import json
import numpy as np
from shapely.geometry import Point, LineString
import requests
import polyline

with open("../data/production/restaurants.json", encoding="utf-8") as f:
    restaurants = json.load(f)

print(f"{len(restaurants)} restaurants geladen")

20 restaurants geladen


/Users/stijnvanbalen/opt/anaconda3/lib/python3.7/site-packages/OpenSSL/SSL.py:15: CryptographyDeprecationWarning: Python 3.7 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.7.
  from cryptography import x509


In [2]:
def get_route(waypoints):
    coords = ";".join([f"{lng},{lat}" for lat, lng in waypoints])
    url = f"http://router.project-osrm.org/route/v1/driving/{coords}"
    params = {"overview": "full", "geometries": "polyline"}
    response = requests.get(url, params=params).json()
    encoded = response["routes"][0]["geometry"]
    return polyline.decode(encoded)

waypoints = [
    (52.39, 4.64),   # Haarlem
    (48.85, 2.35),   # Paris
]

route_coords = get_route(waypoints)
print(f"Route geladen: {len(route_coords)} punten")

Route geladen: 5485 punten


In [4]:
def compute_km_marker(lat, lng, route_coords):
    """
    Zoek het dichtstbijzijnde punt op de route
    en bereken de afstand langs de route tot dat punt.
    """
    min_dist = float("inf")
    closest_idx = 0

    for i, (rlat, rlng) in enumerate(route_coords):
        dist = (lat - rlat) ** 2 + (lng - rlng) ** 2
        if dist < min_dist:
            min_dist = dist
            closest_idx = i

    # Bereken afstand langs de route tot closest_idx
    km = 0.0
    for i in range(1, closest_idx + 1):
        lat1, lng1 = route_coords[i - 1]
        lat2, lng2 = route_coords[i]
        # Haversine benadering: 1 graad lat ≈ 111km, 1 graad lng ≈ 80km op deze breedtegraad
        dlat = (lat2 - lat1) * 111
        dlng = (lng2 - lng1) * 80
        km += (dlat ** 2 + dlng ** 2) ** 0.5

    return round(km, 1)

for r in restaurants:
    r["km_marker"] = compute_km_marker(r["lat"], r["lng"], route_coords)
    print(f"{r['name']:<50} km {r['km_marker']}")

Restaurant Beukenhof Vichte                        km 282.2
't Brigandje                                       km 262.5
't Veer                                            km 270.7
Vijverhof                                          km 299.1
Ferme Balthazar                                    km 293.6
Le Pavé Gourmand                                   km 324.1
RESTAURANT LE STROMBOLI -RESTAURANT-PIZZÉRIA BAPAUME ET ENVIRONS km 386.5
Auberge de la Vallée d'Ancre                       km 389.1
Aux Gars du Nord                                   km 401.1
Le Méditerranée Sarl                               km 405.8
À la Chouette Gourmande                            km 406.5
La Taverne du Cochon Salé                          km 402.5
Le Bistrot D'Antoine                               km 405.8
Le Comptoir de Marius                              km 431.4
A l'Auberge du Bac                                 km 470.9
LE JULIANON                                        km 492.7
Lusitalia Liancourt       

In [5]:
with open("../data/production/restaurants.json", "w", encoding="utf-8") as f:
    json.dump(restaurants, f, indent=2, ensure_ascii=False)

print("Opgeslagen met km_markers")

Opgeslagen met km_markers
